# Demo: Fusion_PSSM1110 测试集混淆矩阵

针对 **最佳方案（1110 维 PSSM 融合 BERT，Fusion_PSSM1110）**，在 **测试集** 上输出混淆矩阵与 AUC/ACC/AUPRC。

- 直接加载 `roc_curve_demo.ipynb` 训练并保存的预测结果（`ours_predictions.npz`）和完整指标（`ours_fusion_metrics.json`），保证各 notebook 之间数值一致。

In [4]:
import os
import json
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, accuracy_score, roc_auc_score, average_precision_score

COMPARISON_RESULTS_DIR = '/home/nemophila/projects/protein_bert/Comparison/results' 


## 加载 Fusion_PSSM1110 预测结果与最优阈值

In [5]:
data = np.load(os.path.join(COMPARISON_RESULTS_DIR, 'ours_predictions.npz'))
y_test = data['y_true']
y_prob_test = data['y_prob']

with open(os.path.join(COMPARISON_RESULTS_DIR, 'ours_fusion_metrics.json')) as f:
    saved = json.load(f)
best_thr = saved['metrics']['Threshold']
y_pred_test = (y_prob_test >= best_thr).astype(int)

print(f'Loaded {len(y_test)} test samples, threshold={best_thr:.4f}')


Loaded 286 test samples, threshold=0.2500


## 测试集混淆矩阵

In [6]:
auc = roc_auc_score(y_test, y_prob_test)
acc = accuracy_score(y_test, y_pred_test)
auprc = average_precision_score(y_test, y_prob_test)
print('Fusion_PSSM1110 (seed=22) — Test set metrics')
print(f'  AUC:   {auc:.4f}')
print(f'  ACC:   {acc:.4f}')
print(f'  AUPRC: {auprc:.4f}')

cm = confusion_matrix(y_test, y_pred_test)
labels = ['Non-Acr', 'Acr']
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
cm_df.index.name = 'True'
cm_df.columns.name = 'Predicted'
print('\nConfusion matrix')
display(cm_df)
